# Pipeline de Normas GRC — Paso 1: Setup y configuración 
  
**Proyecto:** Pipeline de extracción de normas regulatorias colombianas → Landing Zone (HU-NOR-008) 
**Autor:** Esteban Campaz Grisales · Bancolombia — Dirección de Regulación 
**Stack NLP:** 100 % local · sin API externas · solo `pip` 
  
### ¿Qué hace este notebook? 
Crea toda la estructura del proyecto, escribe los archivos de configuración e instala las dependencias. 
  
### Al terminar tendremos: 
- Estructura de carpetas completa (`src/`, `data/`, `logs/`, `notebooks/`) 
- `config.py` — configuración centralizada del pipeline 
- `requirements.txt` — todas las dependencias 
- `.env` — variables de entorno (sin claves externas requeridas) 
- `utils/logger.py` — logger global con rotación de archivos 
- Los tres modelos NLP instalados y verificados 
  
### Stack NLP local 
  
| Tarea | Librería | Modelo | RAM estimada | 
|-------|----------|--------|--------------| 
| Resumen extractivo | `sumy` | LexRank / LSA | ~50 MB | 
| Clasificación por compañía | `sentence-transformers` | MiniLM-L12-v2 multilingual | ~1.5 GB | 
| Extracción de obligaciones | `spaCy` | es_core_news_sm | ~200 MB | 
  
> Los modelos se descargan una sola vez en la primera ejecución y quedan en caché local. 
> A partir de ahí el pipeline funciona sin internet. 


In [1]:
from pathlib import Path 
  
# Por defecto: la carpeta donde está este notebook. 
# Cámbialo si quieres otra ubicación, por ejemplo: 
#   PROJECT_ROOT = Path(r"C:\proyectos\normas_pipeline") 
PROJECT_ROOT = Path.cwd() 
  
print(f"Proyecto se creará en:\n  {PROJECT_ROOT}") 
print("\nSi la ruta no es correcta, cambia PROJECT_ROOT y vuelve a ejecutar esta celda.") 

Proyecto se creará en:
  c:\Users\User\Documents\normas-grc-pipeline-main

Si la ruta no es correcta, cambia PROJECT_ROOT y vuelve a ejecutar esta celda.


In [2]:
CARPETAS = [ 
    "src", "src/scraper", "src/storage", 
    "src/ai", "src/lz", "src/utils", 
    "data", "data/raw", "data/enriched", "data/lz_output", 
    "logs", "notebooks", 
] 
  
for c in CARPETAS: 
    (PROJECT_ROOT / c).mkdir(parents=True, exist_ok=True) 
    print(f"  \u2713 {c}/") 
  
print("\nEstructura de carpetas creada.") 
 


  ✓ src/
  ✓ src/scraper/
  ✓ src/storage/
  ✓ src/ai/
  ✓ src/lz/
  ✓ src/utils/
  ✓ data/
  ✓ data/raw/
  ✓ data/enriched/
  ✓ data/lz_output/
  ✓ logs/
  ✓ notebooks/

Estructura de carpetas creada.


In [3]:
REQUIREMENTS = r'''# ─── Scraping ─── 
requests>=2.31.0 
beautifulsoup4>=4.12.0 
lxml>=5.0.0 
playwright>=1.40.0 
  
# ─── Extracción de PDF ─── 
pdfplumber>=0.10.0 
PyMuPDF>=1.23.0 
  
# ─── Datos / Excel ─── 
openpyxl>=3.1.0 
pandas>=2.1.0 
  
# ─── NLP local (sin API externa, 100 % offline después de la primera descarga) ─── 
sumy>=0.9.0                     # resumen extractivo (LexRank / LSA) 
spacy>=3.7.0                    # tokenización y NER en español 
sentence-transformers>=2.7.0    # clasificación semántica por embeddings 
  
# ─── Infraestructura ─── 
python-dotenv>=1.0.0 
loguru>=0.7.0 
tenacity>=8.2.0 
''' 
(PROJECT_ROOT / 'requirements.txt').write_text(REQUIREMENTS, encoding='utf-8') 
print('\u2713 requirements.txt escrito') 
 


✓ requirements.txt escrito


In [4]:
ENV_EXAMPLE = r'''# ════════════════════════════════════════════════════════ 
#  Variables de entorno — Pipeline de Normas GRC (stack local) 
#  Copia este archivo como .env y ajusta lo que necesites. 
#  No se requiere ninguna API key externa. 
# ════════════════════════════════════════════════════════ 
  
# ─── Rutas ─── 
DATA_PATH=./data 
  
# ─── Logging ─── 
LOG_LEVEL=INFO 
LOG_RETENTION=30 days 
  
# ─── Scraping ─── 
SCRAPER_DELAY_SECONDS=2 
MAX_RETRIES=3 
REQUEST_TIMEOUT=30 
  
# ─── NLP ─── 
SUMY_SENTENCES_COUNT=10 
SPACY_MODEL=es_core_news_sm 
SENTENCE_TRANSFORMER_MODEL=paraphrase-multilingual-MiniLM-L12-v2 
CONFIANZA_MIN=0.65 
''' 
  
(PROJECT_ROOT / '.env.example').write_text(ENV_EXAMPLE, encoding='utf-8') 
print('\u2713 .env.example escrito') 
  
env_path = PROJECT_ROOT / '.env' 
if env_path.exists(): 
    print('\u26a0  .env ya existe — no se sobreescribe') 
else: 
    env_path.write_text(ENV_EXAMPLE, encoding='utf-8') 
    print('\u2713 .env creado') 
print('\nNinguna API key externa requerida.') 
 


✓ .env.example escrito
✓ .env creado

Ninguna API key externa requerida.


In [5]:
GITIGNORE = r'''# Entorno 
.env 
  
# Datos generados 
data/raw/ 
data/enriched/ 
data/lz_output/ 
logs/ 
  
# Python 
__pycache__/ 
*.py[cod] 
.ipynb_checkpoints/ 
.venv/ 
venv/ 
  
# Modelos descargados (se regeneran con pip) 
# ~/.cache/huggingface  ← no está en el proyecto, no hace falta ignorarlo 
''' 
(PROJECT_ROOT / '.gitignore').write_text(GITIGNORE, encoding='utf-8') 
print('\u2713 .gitignore escrito') 
  
 


✓ .gitignore escrito


In [6]:
import base64 
_CONFIG_PY_b64 = 'IiIiCmNvbmZpZy5weSDigJQgQ29uZmlndXJhY2nDs24gY2VudHJhbGl6YWRhIGRlbCBwaXBlbGluZSBkZSBleHRyYWNjacOzbiBkZSBub3JtYXMuCgpTdGFjayBOTFAgMTAwICUgbG9jYWwgKHNpbiBBUEkgZXh0ZXJuYXMpOgogIC0gUmVzdW1lbiAgICAgICAgICA6IHN1bXkgKExleFJhbmsgLyBMU0EpIOKAlCBleHRyYWN0aXZvLCBzaW4gcmVkIG5ldXJvbmFsCiAgLSBDbGFzaWZpY2FjacOzbiAgICA6IHJlZ2xhcyBrZXl3b3JkICsgc2VudGVuY2UtdHJhbnNmb3JtZXJzIChNaW5pTE0gbXVsdGlsaW5ndWFsKQogIC0gT2JsaWdhY2lvbmVzICAgICA6IHNwYUN5ICsgZXNfY29yZV9uZXdzX3NtICsgcGF0cm9uZXMgcmVnZXggbGVnYWxlcwoKTG9zIG1vZGVsb3Mgc2UgZGVzY2FyZ2FuIFVOQSBTT0xBIFZFWiBlbiBsYSBwcmltZXJhIGVqZWN1Y2nDs24geSBxdWVkYW4gZW4gY2FjaMOpCmxvY2FsLiBBIHBhcnRpciBkZSBlbnRvbmNlcyBlbCBwaXBlbGluZSBmdW5jaW9uYSBzaW4gaW50ZXJuZXQuCiIiIgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gZG90ZW52IGltcG9ydCBsb2FkX2RvdGVudgoKIyDilIDilIDilIAgQ2FyZ2EgZGUgLmVudiBqdW50byBhIGVzdGUgYXJjaGl2byDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKQkFTRV9ESVIgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CmxvYWRfZG90ZW52KEJBU0VfRElSIC8gIi5lbnYiKQoKCiMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQCiMgIFJVVEFTCiMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQCkRBVEFfRElSICAgICAgID0gUGF0aChvcy5nZXRlbnYoIkRBVEFfUEFUSCIsIHN0cihCQVNFX0RJUiAvICJkYXRhIikpKQpSQVdfRElSICAgICAgICA9IERBVEFfRElSIC8gInJhdyIKRU5SSUNIRURfRElSICAgPSBEQVRBX0RJUiAvICJlbnJpY2hlZCIKTFpfT1VUUFVUX0RJUiAgPSBEQVRBX0RJUiAvICJsel9vdXRwdXQiCkxPR1NfRElSICAgICAgID0gQkFTRV9ESVIgLyAibG9ncyIKCkFMTF9ESVJTID0gW0RBVEFfRElSLCBSQVdfRElSLCBFTlJJQ0hFRF9ESVIsIExaX09VVFBVVF9ESVIsIExPR1NfRElSXQoKZGVmIGVuc3VyZV9kaXJzKCk6CiAgICAiIiJDcmVhIHRvZGFzIGxhcyBjYXJwZXRhcyBkZSBkYXRvcyB5IGxvZ3Mgc2kgbm8gZXhpc3Rlbi4iIiIKICAgIGZvciBkIGluIEFMTF9ESVJTOgogICAgICAgIGQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKCiMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQCiMgIExPR0dJTkcKIyDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZAKTE9HX0xFVkVMICAgICA9IG9zLmdldGVudigiTE9HX0xFVkVMIiwgIklORk8iKQpMT0dfUkVURU5USU9OID0gb3MuZ2V0ZW52KCJMT0dfUkVURU5USU9OIiwgIjMwIGRheXMiKQoKCiMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQCiMgIFNDUkFQSU5HCiMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQClNDUkFQRVJfREVMQVlfU0VDT05EUyA9IGZsb2F0KG9zLmdldGVudigiU0NSQVBFUl9ERUxBWV9TRUNPTkRTIiwgIjIiKSkKTUFYX1JFVFJJRVMgICAgICAgICAgID0gaW50KG9zLmdldGVudigiTUFYX1JFVFJJRVMiLCAiMyIpKQpSRVFVRVNUX1RJTUVPVVQgICAgICAgPSBpbnQob3MuZ2V0ZW52KCJSRVFVRVNUX1RJTUVPVVQiLCAiMzAiKSkKVVNFUl9BR0VOVCA9IG9zLmdldGVudigKICAgICJVU0VSX0FHRU5UIiwKICAgICJNb3ppbGxhLzUuMCAoY29tcGF0aWJsZTsgTm9ybWFzQm90LzEuMDsgY3VtcGxpbWllbnRvLXJlZ3VsYXRvcmlvKSIsCikKCgojIOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkAojICBOTFAg4oCUIFJFU1VNRU4gKHN1bXkpCiMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQCiMgQWxnb3JpdG1vIGRlIHJlc3VtZW4gZXh0cmFjdGl2by4KIyBMZXhSYW5rIOKGkiBtZWpvciBjYWxpZGFkIGdlbmVyYWwgKGJhc2FkbyBlbiBncmFmb3MgZGUgc2ltaWxpdHVkKS4KIyBMU0EgICAgIOKGkiBtw6FzIHLDoXBpZG8sIGFkZWN1YWRvIHBhcmEgdGV4dG9zIG11eSBsYXJnb3MuClNVTVlfQUxHT1JJVEhNICAgICAgPSBvcy5nZXRlbnYoIlNVTVlfQUxHT1JJVEhNIiwgIkxleFJhbmsiKQpTVU1ZX1NFTlRFTkNFU19DT1VOVCA9IGludChvcy5nZXRlbnYoIlNVTVlfU0VOVEVOQ0VTX0NPVU5UIiwgIjEwIikpClNVTVlfTEFOR1VBR0UgICAgICAgPSAic3BhbmlzaCIKCgojIOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkAojICBOTFAg4oCUIENMQVNJRklDQUNJw5NOIFBPUiBDT01QQcORw41BCiMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQCiMgTW9kZWxvIGRlIGVtYmVkZGluZ3MgbXVsdGlsaW5nw7xlICg0MjAgTUIsIGRlc2NhcmdhIGF1dG9tw6F0aWNhIGxhIDHCqiB2ZXopLgojIENhY2jDqSBlbjogfi8uY2FjaGUvaHVnZ2luZ2ZhY2Uvc2VudGVuY2VfdHJhbnNmb3JtZXJzLwpTRU5URU5DRV9UUkFOU0ZPUk1FUl9NT0RFTCA9IG9zLmdldGVudigKICAgICJTRU5URU5DRV9UUkFOU0ZPUk1FUl9NT0RFTCIsCiAgICAicGFyYXBocmFzZS1tdWx0aWxpbmd1YWwtTWluaUxNLUwxMi12MiIsCikKCiMgVW1icmFsIGRlIGNvbmZpYW56YTogZGViYWpvIOKGkiBwZW5kaWVudGVfdmFsaWRhY2lvbiA9IFRydWUKQ09ORklBTlpBX01JTiA9IGZsb2F0KG9zLmdldGVudigiQ09ORklBTlpBX01JTiIsICIwLjY1IikpCgojIOKUgOKUgCBDYXBhIDE6IHJlZ2xhcyBrZXl3b3JkIGRldGVybWluaXN0YXMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiMgU2kgZWwgdGV4dG8gY29udGllbmUgYWxndW5vIGRlIGVzdG9zIHTDqXJtaW5vcyDihpIgc2UgYXNpZ25hIGVzYSBjb21wYcOxw61hCiMgY29uIGNvbmZpYW56YSAwLjkwIHkgbm8gc2UgcGFzYSBhIGxhIGNhcGEgZGUgZW1iZWRkaW5ncy4KS0VZV09SRF9SVUxFUzogZGljdFtzdHIsIGxpc3Rbc3RyXV0gPSB7CiAgICAiQmFuY29sb21iaWEiOiBbCiAgICAgICAgImVuY2FqZSBiYW5jYXJpbyIsICJjYXJ0ZXJhIGRlIGNyw6lkaXRvIiwgImNhcHRhY2nDs24gZGUgZGVww7NzaXRvcyIsCiAgICAgICAgImN1ZW50YSBjb3JyaWVudGUiLCAiY3VlbnRhIGRlIGFob3Jyb3MiLCAiY3LDqWRpdG8gaGlwb3RlY2FyaW8iLAogICAgICAgICJtaWNyb2Nyw6lkaXRvIiwgImxlYXNpbmcgaGFiaXRhY2lvbmFsIiwgImVzdGFibGVjaW1pZW50byBkZSBjcsOpZGl0byIsCiAgICAgICAgInNpc3RlbWEgZGUgcGFnb3MiLCAiYWN1ZXJkbyBkZSBiYXNpbGVhIiwgImNvZWZpY2llbnRlIGRlIHNvbHZlbmNpYSIsCiAgICAgICAgInByb3Zpc2lvbmVzIGRlIGNhcnRlcmEiLCAicmVmb3JtYSB0cmlidXRhcmlhIiwgImltcHVlc3RvIGRlIHJlbnRhIiwKICAgIF0sCiAgICAiRmlkdWNpYXJpYSI6IFsKICAgICAgICAiZmlkdWNpYSBtZXJjYW50aWwiLCAiZmlkZWljb21pc28iLCAiZm9uZG8gZGUgaW52ZXJzacOzbiBjb2xlY3RpdmEiLAogICAgICAgICIgZmljICIsICJlbmNhcmdvIGZpZHVjaWFyaW8iLCAicGF0cmltb25pbyBhdXTDs25vbW8iLAogICAgICAgICJhZG1pbmlzdHJhY2nDs24gZGUgcmVjdXJzb3MgZGUgdGVyY2Vyb3MiLCAiZm9uZG8gZGUgcGVuc2lvbmVzIHZvbHVudGFyaWFzIiwKICAgICAgICAic29jaWVkYWQgZmlkdWNpYXJpYSIsICJuZWdvY2lvIGZpZHVjaWFyaW8iLAogICAgXSwKICAgICJWYWxvcmVzIjogWwogICAgICAgICJtZXJjYWRvIGRlIHZhbG9yZXMiLCAiY29taXNpb25pc3RhIGRlIGJvbHNhIiwgImJvbHNhIGRlIHZhbG9yZXMgZGUgY29sb21iaWEiLAogICAgICAgICJ0aXR1bGFyaXphY2nDs24iLCAic2VjdXJpdGl6YWNpw7NuIiwgImludGVybWVkaWFjacOzbiBkZSB2YWxvcmVzIiwKICAgICAgICAiYXV0b3JyZWd1bGFkb3IgZGVsIG1lcmNhZG8gZGUgdmFsb3JlcyIsICJhbXYiLCAiY3VzdG9kaWEgZGUgdmFsb3JlcyIsCiAgICAgICAgInBvcnRhZm9saW8gZGUgaW52ZXJzaW9uZXMiLCAic29jaWVkYWQgY29taXNpb25pc3RhIiwgIm9mZXJ0YSBww7pibGljYSIsCiAgICAgICAgInByb3NwZWN0byBkZSBlbWlzacOzbiIsICJhY2Npb25lcyBlbiBib2xzYSIsCiAgICBdLAogICAgIkJhbmNhIGRlIEludmVyc2lvbmVzIjogWwogICAgICAgICJiYW5jYSBkZSBpbnZlcnNpw7NuIiwgImZ1c2nDs24geSBhZHF1aXNpY2nDs24iLCAibSZhIiwKICAgICAgICAiYXNlc29yw61hIGZpbmFuY2llcmEgY29ycG9yYXRpdmEiLCAiZXN0cnVjdHVyYWNpw7NuIGRlIGRldWRhIiwKICAgICAgICAidW5kZXJ3cml0aW5nIiwgInByb2plY3QgZmluYW5jZSIsICJzaW5kaWNhY2nDs24gZGUgY3LDqWRpdG9zIiwKICAgICAgICAiY2FwaXRhbCBwcml2YWRvIiwgInByaXZhdGUgZXF1aXR5IiwKICAgIF0sCn0KCiMg4pSA4pSAIENhcGEgMjogdGV4dG9zIHNlbWlsbGEgcGFyYSBlbWJlZGRpbmdzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAojIHNlbnRlbmNlLXRyYW5zZm9ybWVycyBjYWxjdWxhIHNpbWlsaXR1ZCBlbnRyZSBlbCB0ZXh0byBkZSBsYSBub3JtYQojIHkgY2FkYSB0ZXh0byBzZW1pbGxhOyBhc2lnbmEgbGEgY29tcGHDscOtYSBkZSBtYXlvciBzaW1pbGl0dWQuCkNPTVBBTllfU0VFRF9URVhUUzogZGljdFtzdHIsIHN0cl0gPSB7CiAgICAiQmFuY29sb21iaWEiOiAoCiAgICAgICAgIm5vcm1hcyBxdWUgcmVndWxhbiBvcGVyYWNpb25lcyBkZSBjcsOpZGl0bywgY2FwdGFjacOzbiBkZSBkZXDDs3NpdG9zLCAiCiAgICAgICAgImVuY2FqZSBiYW5jYXJpbywgc29sdmVuY2lhIHkgZXN0YWJsZWNpbWllbnRvcyBkZSBjcsOpZGl0byBlbiBDb2xvbWJpYSIKICAgICksCiAgICAiRmlkdWNpYXJpYSI6ICgKICAgICAgICAibm9ybWFzIHF1ZSByZWd1bGFuIGZvbmRvcyBkZSBpbnZlcnNpw7NuIGNvbGVjdGl2YSwgZmlkZWljb21pc29zLCAiCiAgICAgICAgImVuY2FyZ29zIGZpZHVjaWFyaW9zLCBwYXRyaW1vbmlvcyBhdXTDs25vbW9zIHkgc29jaWVkYWRlcyBmaWR1Y2lhcmlhcyIKICAgICksCiAgICAiVmFsb3JlcyI6ICgKICAgICAgICAibm9ybWFzIHF1ZSByZWd1bGFuIGVsIG1lcmNhZG8gZGUgdmFsb3JlcywgY29taXNpb25pc3RhcyBkZSBib2xzYSwgIgogICAgICAgICJpbnRlcm1lZGlhY2nDs24gYnVyc8OhdGlsLCBjdXN0b2RpYSBkZSB2YWxvcmVzIHkgbGEgYm9sc2EgZGUgQ29sb21iaWEiCiAgICApLAogICAgIkJhbmNhIGRlIEludmVyc2lvbmVzIjogKAogICAgICAgICJub3JtYXMgcXVlIHJlZ3VsYW4gbGEgYmFuY2EgZGUgaW52ZXJzacOzbiwgZnVzaW9uZXMgeSBhZHF1aXNpY2lvbmVzLCAiCiAgICAgICAgImFzZXNvcsOtYSBmaW5hbmNpZXJhIGNvcnBvcmF0aXZhIHkgZXN0cnVjdHVyYWNpw7NuIGRlIGNhcGl0YWwiCiAgICApLAp9CgoKIyDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZAKIyAgTkxQIOKAlCBFWFRSQUNDScOTTiBERSBPQkxJR0FDSU9ORVMgKHNwYUN5ICsgcmVnZXgpCiMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQClNQQUNZX01PREVMID0gb3MuZ2V0ZW52KCJTUEFDWV9NT0RFTCIsICJlc19jb3JlX25ld3Nfc20iKQoKIyBQYXRyb25lcyByZWdleCBxdWUgc2XDsWFsYW4gZWwgaW5pY2lvIGRlIHVuYSBvYmxpZ2FjacOzbiBsZWdhbC4KIyBVbmEgb3JhY2nDs24gcXVlIGNvbnRlbmdhIGFsZ3VubyBkZSBlc3RvcyBtYXJjYWRvcmVzIHNlIGNhbmRpZGF0YQojIGNvbW8gb2JsaWdhY2nDs24uIE9yZGVuYWRvcyBkZSBtw6FzIGEgbWVub3MgZXNwZWPDrWZpY28uCk9CTElHQVRJT05fTUFSS0VSUzogbGlzdFtzdHJdID0gWwogICAgciIIZGViZXLDoQgiLAogICAgciIIZGViZXLDoW4IIiwKICAgIHIiCGRlYmUIIiwKICAgIHIiCGRlYmVuCCIsCiAgICByIghlc3TDoSBvYmxpZ2FkbwgiLAogICAgciIIZXN0w6FuIG9ibGlnYWRvcwgiLAogICAgciIIcXVlZGEgb2JsaWdhZG8IIiwKICAgIHIiCHNlIHByb2jDrWJlCCIsCiAgICByIghxdWVkYSBwcm9oaWJpZG8IIiwKICAgIHIiCGVuIG5pbmfDum4gY2FzbwgiLAogICAgciIIZXMgb2JsaWdhdG9yaW8IIiwKICAgIHIiCHNlcsOhIG9ibGlnYXRvcmlvCCIsCiAgICByIghzZSBleGlnZQgiLAogICAgciIIZXMgcmVxdWlzaXRvCCIsCiAgICByIghkZW50cm8gZGVsIHBsYXpvIGRlCCIsCiAgICByIghlbiB1biBwbGF6byBubyBtYXlvcggiLAogICAgciIIc28gcGVuYSBkZQgiLAogICAgciIIYmFqbyBwZW5hIGRlCCIsCl0KCiMgTsO6bWVybyBtw6F4aW1vIGRlIG9ibGlnYWNpb25lcyBhIGV4dHJhZXIgcG9yIG5vcm1hCk1BWF9PQkxJR0FDSU9ORVNfUE9SX05PUk1BID0gNTAwCgoKIyDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZAKIyAgRlVFTlRFUyBSRUdVTEFUT1JJQVMKIyDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZAKU09VUkNFUzogZGljdFtzdHIsIGRpY3RdID0gewogICAgIlNGQyI6IHsKICAgICAgICAibm9tYnJlIjogIlN1cGVyaW50ZW5kZW5jaWEgRmluYW5jaWVyYSBkZSBDb2xvbWJpYSIsCiAgICAgICAgInVybF9iYXNlIjogImh0dHBzOi8vd3d3LnN1cGVyZmluYW5jaWVyYS5nb3YuY28iLAogICAgICAgICJ1cmxfaW5kZXgiOiAiIiwgICAgICAgICAgIyBUT0RPIEZhc2UgMgogICAgICAgICJ0aXBvX2NvbnRlbmlkbyI6ICJtaXh0byIsCiAgICAgICAgImFjdGl2byI6IFRydWUsCiAgICB9LAogICAgIkJBTlJFUCI6IHsKICAgICAgICAibm9tYnJlIjogIkJhbmNvIGRlIGxhIFJlcMO6YmxpY2EiLAogICAgICAgICJ1cmxfYmFzZSI6ICJodHRwczovL3d3dy5iYW5yZXAuZ292LmNvIiwKICAgICAgICAidXJsX2luZGV4IjogIiIsCiAgICAgICAgInRpcG9fY29udGVuaWRvIjogIm1peHRvIiwKICAgICAgICAiYWN0aXZvIjogVHJ1ZSwKICAgIH0sCiAgICAiTUlOSEFDIjogewogICAgICAgICJub21icmUiOiAiTWluaXN0ZXJpbyBkZSBIYWNpZW5kYSB5IENyw6lkaXRvIFDDumJsaWNvIiwKICAgICAgICAidXJsX2Jhc2UiOiAiaHR0cHM6Ly93d3cubWluaGFjaWVuZGEuZ292LmNvIiwKICAgICAgICAidXJsX2luZGV4IjogIiIsCiAgICAgICAgInRpcG9fY29udGVuaWRvIjogIm1peHRvIiwKICAgICAgICAiYWN0aXZvIjogVHJ1ZSwKICAgIH0sCiAgICAiRElBTiI6IHsKICAgICAgICAibm9tYnJlIjogIkRpcmVjY2nDs24gZGUgSW1wdWVzdG9zIHkgQWR1YW5hcyBOYWNpb25hbGVzIiwKICAgICAgICAidXJsX2Jhc2UiOiAiaHR0cHM6Ly93d3cuZGlhbi5nb3YuY28iLAogICAgICAgICJ1cmxfaW5kZXgiOiAiIiwKICAgICAgICAidGlwb19jb250ZW5pZG8iOiAibWl4dG8iLAogICAgICAgICJhY3Rpdm8iOiBUcnVlLAogICAgfSwKICAgICJVUkYiOiB7CiAgICAgICAgIm5vbWJyZSI6ICJVbmlkYWQgZGUgUmVndWxhY2nDs24gRmluYW5jaWVyYSIsCiAgICAgICAgInVybF9iYXNlIjogImh0dHBzOi8vd3d3LnVyZi5nb3YuY28iLAogICAgICAgICJ1cmxfaW5kZXgiOiAiIiwKICAgICAgICAidGlwb19jb250ZW5pZG8iOiAibWl4dG8iLAogICAgICAgICJhY3Rpdm8iOiBGYWxzZSwKICAgIH0sCiAgICAiQ09OR1JFU08iOiB7CiAgICAgICAgIm5vbWJyZSI6ICJDb25ncmVzbyBkZSBsYSBSZXDDumJsaWNhIiwKICAgICAgICAidXJsX2Jhc2UiOiAiaHR0cDovL3d3dy5zZWNyZXRhcmlhc2VuYWRvLmdvdi5jbyIsCiAgICAgICAgInVybF9pbmRleCI6ICIiLAogICAgICAgICJ0aXBvX2NvbnRlbmlkbyI6ICJodG1sIiwKICAgICAgICAiYWN0aXZvIjogRmFsc2UsCiAgICB9LAogICAgIkFNViI6IHsKICAgICAgICAibm9tYnJlIjogIkF1dG9ycmVndWxhZG9yIGRlbCBNZXJjYWRvIGRlIFZhbG9yZXMiLAogICAgICAgICJ1cmxfYmFzZSI6ICJodHRwczovL3d3dy5hbXZjb2xvbWJpYS5vcmcuY28iLAogICAgICAgICJ1cmxfaW5kZXgiOiAiIiwKICAgICAgICAidGlwb19jb250ZW5pZG8iOiAiaHRtbCIsCiAgICAgICAgImFjdGl2byI6IEZhbHNlLAogICAgfSwKICAgICJTVVBFUlNPQyI6IHsKICAgICAgICAibm9tYnJlIjogIlN1cGVyaW50ZW5kZW5jaWEgZGUgU29jaWVkYWRlcyIsCiAgICAgICAgInVybF9iYXNlIjogImh0dHBzOi8vd3d3LnN1cGVyc29jaWVkYWRlcy5nb3YuY28iLAogICAgICAgICJ1cmxfaW5kZXgiOiAiIiwKICAgICAgICAidGlwb19jb250ZW5pZG8iOiAibWl4dG8iLAogICAgICAgICJhY3Rpdm8iOiBGYWxzZSwKICAgIH0sCn0KCmRlZiBmdWVudGVzX2FjdGl2YXMoKSAtPiBkaWN0OgogICAgIiIiRGV2dWVsdmUgc29sbyBsYXMgZnVlbnRlcyBtYXJjYWRhcyBjb21vIGFjdGl2YXMuIiIiCiAgICByZXR1cm4ge2s6IHYgZm9yIGssIHYgaW4gU09VUkNFUy5pdGVtcygpIGlmIHYuZ2V0KCJhY3Rpdm8iKX0KCgojIOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkAojICBDQVTDgUxPR09TIERFTCBORUdPQ0lPCiMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQCkNPTVBBTklBUzogbGlzdFtzdHJdID0gWwogICAgIkJhbmNvbG9tYmlhIiwgIkZpZHVjaWFyaWEiLCAiVmFsb3JlcyIsICJCYW5jYSBkZSBJbnZlcnNpb25lcyIKXQoKVElQT1NfTk9STUE6IGxpc3Rbc3RyXSA9IFsKICAgICJDb25zdGl0dWNpw7NuIiwgIkxleSIsICJEZWNyZXRvIiwgIkFjdWVyZG9zIiwKICAgICJSZWdsYW1lbnRvcyIsICJSZXNvbHVjacOzbiIsICJDaXJjdWxhcmVzIiwKXQoKCiMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQCiMgIFNDSEVNQSBERSBMQSBMQU5ESU5HIFpPTkUgKGNvbnNpc3RlbnRlIGNvbiBIVS1OT1ItMDA4KQojIOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkApTQ0hFTUFfTFpfTk9STUE6IGxpc3Rbc3RyXSA9IFsKICAgICJpZF9leHRlcm5vX2x6IiwgImZ1ZW50ZSIsICJub21icmUiLCAidGlwb19ub3JtYSIsCiAgICAiYXV0b3JpZGFkX2VtaXRlIiwgImF1dG9yaWRhZF92aWdpbGEiLCAiZmVjaGFfZXhwZWRpY2lvbiIsCiAgICAiZmVjaGFfZXh0cmFjY2lvbiIsICJkZXNjcmlwY2lvbl9ubHAiLCAidXJsIiwgInRleHRvX29yaWdpbmFsIiwKICAgICJvcmlnZW5fY3JlYWNpb24iLCAiZXN0YWRvX2NsYXNpZmljYWNpb24iLAogICAgImNvbmZpYW56YV9jbGFzaWZpY2FjaW9uIiwgInBlbmRpZW50ZV92YWxpZGFjaW9uIiwKXQoKU0NIRU1BX0xaX05PUk1BX0NPTVBBTklBOiBsaXN0W3N0cl0gPSBbCiAgICAiaWRfZXh0ZXJub19seiIsICJjb21wYW5pYSIsICJlc19wcmluY2lwYWwiCl0KClNDSEVNQV9MWl9PQkxJR0FDSU9OOiBsaXN0W3N0cl0gPSBbCiAgICAiaWRfZXh0ZXJub19sel9ub3JtYSIsICJudW1lcmFsIiwgImZlY2hhX2VudHJhZGEiLAogICAgInRleHRvX29ibGlnYWNpb24iLCAiY29uZmlhbnphX2V4dHJhY2Npb24iLApdCgpTQ0hFTUFfUkFXOiBsaXN0W3N0cl0gPSBbCiAgICAiaWRfZXh0ZXJubyIsICJmdWVudGUiLCAibm9tYnJlIiwgInRpcG9fbm9ybWEiLCAiYXV0b3JpZGFkIiwKICAgICJmZWNoYV9leHBlZGljaW9uIiwgImZlY2hhX2V4dHJhY2Npb24iLCAidXJsIiwKICAgICJ0ZXh0b19jb21wbGV0byIsICJoYXNoX2NvbnRlbmlkbyIsCl0KCgojIOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkAojICBMw41NSVRFUyAoY29uc2lzdGVudGVzIGNvbiBsYXMgSFVzKQojIOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkOKVkApNQVhfTk9NQlJFICAgICAgICAgICAgICA9IDIwMApNQVhfREVTQ1JJUENJT04gICAgICAgICA9IDUwMDAKTUFYX1RFWFRPX09CTElHQUNJT04gICAgPSA1MDAwCk1BWF9VUkwgICAgICAgICAgICAgICAgID0gNTAwCk1BWF9OVU1FUkFMICAgICAgICAgICAgID0gMTAK' 
CONFIG_PY = base64.b64decode(_CONFIG_PY_b64).decode('utf-8') 
(PROJECT_ROOT / 'config.py').write_text(CONFIG_PY, encoding='utf-8') 
print('\u2713 config.py escrito ({:.1f} KB)'.format(len(CONFIG_PY)/1024))


✓ config.py escrito (10.0 KB)


In [7]:
import base64 
_log_b64 = 'IiIiCnV0aWxzL2xvZ2dlci5weSDigJQgTG9nZ2VyIGdsb2JhbCBkZWwgcGlwZWxpbmUsIGJhc2FkbyBlbiBsb2d1cnUuCgpVc28gZW4gY3VhbHF1aWVyIG3Ds2R1bG86CiAgICBmcm9tIHV0aWxzLmxvZ2dlciBpbXBvcnQgZ2V0X2xvZ2dlcgogICAgbG9nID0gZ2V0X2xvZ2dlcigic2ZjX3NjcmFwZXIiKQogICAgbG9nLmluZm8oIlByb2Nlc2FuZG8gbm9ybWEge30iLCBub21icmUpCgpTYWxpZGFzOgogIC0gQ29uc29sYSAobml2ZWwgY29uZmlndXJhYmxlIGNvbiBMT0dfTEVWRUwgZW4gLmVudikuCiAgLSBBcmNoaXZvIGxvZ3MvcGlwZWxpbmVfWVlZWU1NREQubG9nIGNvbiByb3RhY2nDs24gZGlhcmlhLgoiIiIKaW1wb3J0IHN5cwpmcm9tIGxvZ3VydSBpbXBvcnQgbG9nZ2VyCmltcG9ydCBjb25maWcKCmNvbmZpZy5lbnN1cmVfZGlycygpCgpsb2dnZXIuY29uZmlndXJlKGV4dHJhPXsibW9kdWxlIjogInBpcGVsaW5lIn0pCmxvZ2dlci5yZW1vdmUoKQoKbG9nZ2VyLmFkZCgKICAgIHN5cy5zdGRlcnIsCiAgICBsZXZlbD1jb25maWcuTE9HX0xFVkVMLAogICAgZm9ybWF0PSgKICAgICAgICAiPGdyZWVuPnt0aW1lOkhIOm1tOnNzfTwvZ3JlZW4+IHwgIgogICAgICAgICI8bGV2ZWw+e2xldmVsOiA8OH08L2xldmVsPiB8ICIKICAgICAgICAiPGN5YW4+e2V4dHJhW21vZHVsZV19PC9jeWFuPiB8ICIKICAgICAgICAie21lc3NhZ2V9IgogICAgKSwKKQoKbG9nZ2VyLmFkZCgKICAgIGNvbmZpZy5MT0dTX0RJUiAvICJwaXBlbGluZV97dGltZTpZWVlZTU1ERH0ubG9nIiwKICAgIGxldmVsPSJERUJVRyIsCiAgICByb3RhdGlvbj0iMDA6MDAiLAogICAgcmV0ZW50aW9uPWNvbmZpZy5MT0dfUkVURU5USU9OLAogICAgZW5jb2Rpbmc9InV0Zi04IiwKICAgIGZvcm1hdD0ie3RpbWU6WVlZWS1NTS1ERCBISDptbTpzc30gfCB7bGV2ZWw6IDw4fSB8IHtleHRyYVttb2R1bGVdfSB8IHttZXNzYWdlfSIsCikKCgpkZWYgZ2V0X2xvZ2dlcihtb2R1bGU6IHN0cik6CiAgICAiIiJEZXZ1ZWx2ZSB1biBsb2dnZXIgY29uIGVsIGNhbXBvIG1vZHVsZSBlbmxhemFkbyBhbCBub21icmUgZGFkby4iIiIKICAgIHJldHVybiBsb2dnZXIuYmluZChtb2R1bGU9bW9kdWxlKQo=' 
LOGGER_PY = base64.b64decode(_log_b64).decode('utf-8') 
(PROJECT_ROOT / 'src' / 'utils' / 'logger.py').write_text(LOGGER_PY, encoding='utf-8') 
print('\u2713 src/utils/logger.py') 
  
_init_b64 = 'IiIiUGFxdWV0ZSBkZWwgcGlwZWxpbmUgZGUgZXh0cmFjY2nDs24gZGUgbm9ybWFzLiIiIgo=' 
INIT_PY = base64.b64decode(_init_b64).decode('utf-8') 
paquetes = ['src', 'src/scraper', 'src/storage', 'src/ai', 'src/lz', 'src/utils'] 
for p in paquetes: 
    (PROJECT_ROOT / p / '__init__.py').write_text(INIT_PY, encoding='utf-8') 
    print(f'  \u2713 {p}/__init__.py')


✓ src/utils/logger.py
  ✓ src/__init__.py
  ✓ src/scraper/__init__.py
  ✓ src/storage/__init__.py
  ✓ src/ai/__init__.py
  ✓ src/lz/__init__.py
  ✓ src/utils/__init__.py


In [8]:
import sys 
print("Instalando dependencias...") 
! "{sys.executable}" -m pip install requests beautifulsoup4 lxml pdfplumber PyMuPDF openpyxl pandas sumy spacy sentence-transformers python-dotenv loguru tenacity --quiet
print("\nDependencias instaladas.") 


Instalando dependencias...

Dependencias instaladas.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [9]:
import sys 
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    print ("Skealen dispoible - clasificador TF-IDF listo")
    print ("Sin descargas")

except ImportError:
    import subprocess
    subprocess.run([sys.executable, "m", "pip", "install", "scikit-learn", "--quiet"])
    print ("scikit-learn instalado")

    print("\n Clasificador : reglas keyword (capa 1) + TF-IDF coseno (Capa2)")
    print("100% local, sin modelos externos, sin internet")


Skealen dispoible - clasificador TF-IDF listo
Sin descargas


In [10]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "es-core-news-sm", "--quiet"])

import spacy
nlp = spacy.load("es_core_news_sm")
print("Spacy cargo correctamente")

OSError: [E050] Can't find model 'es_core_news_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

In [ ]:
import sys, importlib 
from pathlib import Path 
  
for ruta in (str(PROJECT_ROOT), str(PROJECT_ROOT / "src")): 
    if ruta not in sys.path: 
        sys.path.insert(0, ruta) 
  
# ── Config ──────────────────────────────────────────────── 
import config 
importlib.reload(config) 
config.ensure_dirs() 
  
# ── Logger ──────────────────────────────────────────────── 
from utils.logger import get_logger 
log = get_logger("setup") 
log.info("Logger funcionando correctamente.") 
  
# ── spaCy ───────────────────────────────────────────────── 
import spacy 
nlp = spacy.load(config.SPACY_MODEL) 
doc = nlp("El banco deberá presentar el informe de solvencia dentro del plazo establecido.") 
tokens = [t.text for t in doc if not t.is_stop and not t.is_punct] 
log.info(f"spaCy OK — {len(list(doc.sents))} oracion(es), {len(tokens)} tokens relevantes") 
  
# ── sumy ────────────────────────────────────────────────── 
from sumy.parsers.plaintext import PlaintextParser 
from sumy.nlp.tokenizers import Tokenizer 
from sumy.summarizers.lex_rank import LexRankSummarizer 
_texto_prueba = ( 
    "La Superintendencia Financiera establece nuevos requisitos de capital. " 
    "Los establecimientos de crédito deberán mantener un índice de solvencia mínimo del 9 por ciento. " 
    "Esta norma aplica a todos los bancos y corporaciones financieras del país. " 
    "El incumplimiento acarreará sanciones administrativas y económicas. " 
    "Los reportes deben enviarse trimestralmente al ente regulador." 
) 
_parser = PlaintextParser.from_string(_texto_prueba, Tokenizer("spanish")) 
_summ   = LexRankSummarizer() 
_result = _summ(_parser.document, sentences_count=2) 
log.info(f"sumy OK — resumen de {len(_result)} oración(es) extraída(s)") 
  
# ── Resumen de config ───────────────────────────────────── 
print("\n" + "=" * 58) 
print("  VERIFICACIÓN COMPLETA — PASO 1") 
print("=" * 58) 
print(f"  BASE_DIR               : {config.BASE_DIR}") 
print(f"  DATA_DIR               : {config.DATA_DIR}") 
print(f"  Fuentes activas        : {', '.join(config.fuentes_activas().keys())}") 
print(f"  Modelo spaCy           : {config.SPACY_MODEL}  \u2713") 
print(f"  Modelo embeddings      : {config.SENTENCE_TRANSFORMER_MODEL}") 
print(f"  Algoritmo resumen      : {config.SUMY_ALGORITHM} ({config.SUMY_SENTENCES_COUNT} oraciones)") 
print(f"  Umbral confianza       : {config.CONFIANZA_MIN}") 
print(f"  Marcadores obligación  : {len(config.OBLIGATION_MARKERS)} patrones") 
print(f"  Reglas keyword         : {sum(len(v) for v in config.KEYWORD_RULES.values())} términos en {len(config.KEYWORD_RULES)} compañías") 
print(f"  Schema LZ norma        : {len(config.SCHEMA_LZ_NORMA)} columnas") 
print(f"  Schema LZ obligación   : {len(config.SCHEMA_LZ_OBLIGACION)} columnas") 
print("=" * 58) 
print("\n  spaCy       \u2713") 
print("  sumy        \u2713") 
print("  sentence-transformers  (verificado en celda 10)") 
print("  logger      \u2713") 
print("\n  Paso 1 completo. Puedes pasar al Paso 2 (Scraper).") 
 


21:21:46 | INFO     | setup | Logger funcionando correctamente.
21:21:46 | INFO     | setup | spaCy OK — 1 oracion(es), 7 tokens relevantes
21:21:50 | INFO     | setup | sumy OK — resumen de 2 oración(es) extraída(s)



  VERIFICACIÓN COMPLETA — PASO 1
  BASE_DIR               : C:\Users\ecampaz\Documents\WebScrapping
  DATA_DIR               : data
  Fuentes activas        : SFC, BANREP, MINHAC, DIAN
  Modelo spaCy           : es_core_news_sm  ✓
  Modelo embeddings      : paraphrase-multilingual-MiniLM-L12-v2
  Algoritmo resumen      : LexRank (10 oraciones)
  Umbral confianza       : 0.65
  Marcadores obligación  : 18 patrones
  Reglas keyword         : 49 términos en 4 compañías
  Schema LZ norma        : 15 columnas
  Schema LZ obligación   : 5 columnas

  spaCy       ✓
  sumy        ✓
  sentence-transformers  (verificado en celda 10)
  logger      ✓

  Paso 1 completo. Puedes pasar al Paso 2 (Scraper).
